In [1]:

from pathlib import Path
import json
import pandas as pd

# --------------------------------------------------
# 1. Configure paths
# --------------------------------------------------

SCRIPT_DIR = Path(__file__).resolve().parent
WORKSPACE_DIR = SCRIPT_DIR.parent
DATA_DIR = WORKSPACE_DIR / "XES3G5M"
REPORT_DIR = SCRIPT_DIR / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("CFLP DATASET AUDIT")
print("=" * 65)
print("Dataset directory:", DATA_DIR)
print("Report directory:", REPORT_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATA_DIR}\n"
        "Place CFLP_Project beside your existing XES3G5M folder."
    )

# --------------------------------------------------
# 2. Inventory dataset files
# --------------------------------------------------

print("\n[1] DATASET FILE INVENTORY")

for path in sorted(DATA_DIR.rglob("*")):
    if path.is_file():
        relative = path.relative_to(DATA_DIR)
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{relative} | {size_mb:.2f} MB")

# --------------------------------------------------
# 3. Inspect CSV schemas and sample rows
# --------------------------------------------------

csv_paths = [
    DATA_DIR / "kc_level" / "train_valid_sequences.csv",
    DATA_DIR / "kc_level" / "test.csv",
    DATA_DIR / "kc_level" / "test_question_window_sequences.csv",
    DATA_DIR / "question_level" / "test_quelevel.csv",
    DATA_DIR / "question_level" /
        "train_valid_sequences_quelevel.csv",
]

print("\n[2] CSV SCHEMA INSPECTION")

csv_report = {}

for path in csv_paths:
    if not path.exists():
        continue

    print(f"\nFile: {path.relative_to(DATA_DIR)}")

    # Read just the header first.
    header = pd.read_csv(path, nrows=0)
    print("Columns:", header.columns.tolist())

    # Read at most five rows for initial inspection.
    sample = pd.read_csv(path, nrows=5)

    print("Sample shape:", sample.shape)
    print(sample.head().to_string(index=False)[:2500])

    csv_report[str(path.relative_to(DATA_DIR))] = {
        "columns": header.columns.tolist(),
        "sample_rows": len(sample),
    }

# --------------------------------------------------
# 4. Inspect question metadata
# --------------------------------------------------

print("\n[3] QUESTION METADATA")

question_path = DATA_DIR / "metadata" / "questions.json"

if question_path.exists():
    with question_path.open("r", encoding="utf-8") as file:
        questions = json.load(file)

    print("Container type:", type(questions).__name__)

    if isinstance(questions, dict):
        print("Question count:", len(questions))
        first_key = next(iter(questions), None)

        if first_key is not None:
            print("Example question ID:", first_key)
            print(json.dumps(
                questions[first_key],
                ensure_ascii=False,
                indent=2
            )[:2500])

    elif isinstance(questions, list):
        print("Question count:", len(questions))

        if questions:
            print(json.dumps(
                questions[0],
                ensure_ascii=False,
                indent=2
            )[:2500])
else:
    print("questions.json not found.")

# --------------------------------------------------
# 5. Inspect original and English KC maps
# --------------------------------------------------

print("\n[4] KC MAP VALIDATION")

map_paths = [
    DATA_DIR / "metadata" / "kc_routes_map.json",
    DATA_DIR / "metadata" / "kc_routes_map_en.json",
]

kc_maps = {}

for path in map_paths:
    if not path.exists():
        print("Missing:", path.name)
        continue

    with path.open("r", encoding="utf-8") as file:
        kc_map = json.load(file)

    kc_maps[path.name] = kc_map
    print(f"{path.name}: {len(kc_map)} entries")

if len(kc_maps) == 2:
    original = kc_maps["kc_routes_map.json"]
    english = kc_maps["kc_routes_map_en.json"]

    original_ids = set(original.keys())
    english_ids = set(english.keys())

    print("IDs missing from English map:",
          len(original_ids - english_ids))
    print("IDs missing from original map:",
          len(english_ids - original_ids))

    blank_translations = [
        key for key, value in english.items()
        if not isinstance(value, str) or not value.strip()
    ]

    print("Blank/invalid English values:",
          len(blank_translations))

# --------------------------------------------------
# 6. Save initial report
# --------------------------------------------------

report_path = REPORT_DIR / "dataset_audit_initial.json"

with report_path.open("w", encoding="utf-8") as file:
    json.dump(csv_report, file, indent=2, ensure_ascii=False)

print("\n" + "=" * 65)
print("INITIAL AUDIT COMPLETE")
print("Report saved to:", report_path)
print("=" * 65)

NameError: name '__file__' is not defined